# Part 1: Real Life Examples and Simpson's Paradox

This notebook addresses Part 1a (confounders, colliders, mediation) and Part 1b (Simpson's paradox).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from causalgraphicalmodels import CausalGraphicalModel
import statsmodels.api as sm
from scipy import stats
import os

# Set random seed for reproducibility
np.random.seed(42)
plt.style.use('seaborn-v0_8')

# Create output directory if it doesn't exist
output_dir = '../output'
os.makedirs(output_dir, exist_ok=True)

## Part 1a: Real Life Examples (2 points)

### Confounders, Colliders, and Mediation

### 1. Confounder

**Definition**: A confounder is a variable that influences both the treatment/exposure and the outcome, creating a spurious association between them.

**Economic Example**: The relationship between education (X) and income (Y), with ability (Z) as a confounder.
- Education (X): Years of schooling
- Income (Y): Annual salary
- Ability (Z): Innate cognitive ability

Ability affects both education (smarter people tend to get more education) and income (smarter people tend to earn more), creating a spurious correlation between education and income if we don't control for ability.

In [ ]:
# Create and visualize confounder DAG
confounder_dag = CausalGraphicalModel(
    nodes=['Ability', 'Education', 'Income'],
    edges=[
        ('Ability', 'Education'),
        ('Ability', 'Income'),
        ('Education', 'Income')
    ]
)

# Draw the DAG
plt.figure(figsize=(8, 6))
confounder_dag.draw()
plt.title('Confounder Example: Education, Income, and Ability')
plt.savefig(f'{output_dir}/confounder_dag.png', dpi=300, bbox_inches='tight')
plt.show()

### 2. Collider

**Definition**: A collider is a variable that is caused by both the treatment/exposure and another variable. Controlling for a collider can introduce bias.

**Economic Example**: The relationship between talent (X) and success (Y), with graduate school admission (Z) as a collider.
- Talent (X): Natural ability in a field
- Success (Y): Career achievement
- Graduate School (Z): Whether person attended graduate school

Both talent and success influence graduate school admission. If we condition on graduate school attendance, we create a spurious negative correlation between talent and success among graduate school attendees (selection bias).

In [ ]:
# Create and visualize collider DAG
collider_dag = CausalGraphicalModel(
    nodes=['Talent', 'Success', 'GradSchool'],
    edges=[
        ('Talent', 'GradSchool'),
        ('Success', 'GradSchool'),
        ('Talent', 'Success')
    ]
)

plt.figure(figsize=(8, 6))
collider_dag.draw()
plt.title('Collider Example: Talent, Success, and Graduate School')
plt.savefig(f'{output_dir}/collider_dag.png', dpi=300, bbox_inches='tight')
plt.show()

### 3. Mediation

**Definition**: A mediator is a variable that lies on the causal path between the treatment and outcome, transmitting part of the treatment's effect.

**Economic Example**: The relationship between minimum wage increases (X) and unemployment (Y), with labor costs (Z) as a mediator.
- Minimum Wage (X): Policy increase in minimum wage
- Unemployment (Y): Regional unemployment rate
- Labor Costs (Z): Total labor costs for firms

Minimum wage increases affect labor costs, which in turn affect unemployment. Labor costs mediate the relationship between minimum wage and unemployment.

In [ ]:
# Create and visualize mediation DAG
mediation_dag = CausalGraphicalModel(
    nodes=['MinWage', 'LaborCosts', 'Unemployment'],
    edges=[
        ('MinWage', 'LaborCosts'),
        ('LaborCosts', 'Unemployment'),
        ('MinWage', 'Unemployment')  # Direct effect
    ]
)

plt.figure(figsize=(8, 6))
mediation_dag.draw()
plt.title('Mediation Example: Minimum Wage, Labor Costs, and Unemployment')
plt.savefig(f'{output_dir}/mediation_dag.png', dpi=300, bbox_inches='tight')
plt.show()

## Part 1b: Simpson's Paradox (2 points)

**Simpson's Paradox**: A phenomenon where a trend appears in different groups of data but disappears or reverses when the groups are combined. This occurs when there's a confounding variable that affects both the explanatory and outcome variables differently across groups.

In [ ]:
# Simulate Simpson's Paradox data
n_per_group = 200

# Group 1: High-skill workers
# X (training hours) and Y (productivity) have positive relationship
X1 = np.random.normal(8, 1, n_per_group)  # More training hours
Y1 = 50 + 2 * X1 + np.random.normal(0, 3, n_per_group)  # Higher base productivity
group1 = np.ones(n_per_group)

# Group 2: Low-skill workers  
# X (training hours) and Y (productivity) have positive relationship
X2 = np.random.normal(4, 1, n_per_group)  # Fewer training hours
Y2 = 20 + 2 * X2 + np.random.normal(0, 3, n_per_group)  # Lower base productivity
group2 = np.zeros(n_per_group)

# Combine groups
X_combined = np.concatenate([X1, X2])
Y_combined = np.concatenate([Y1, Y2])
groups = np.concatenate([group1, group2])

# Create DataFrame
df = pd.DataFrame({
    'training_hours': X_combined,
    'productivity': Y_combined,
    'skill_group': groups,
    'skill_label': ['High-skill' if g == 1 else 'Low-skill' for g in groups]
})

print("Data summary:")
print(df.groupby('skill_label')[['training_hours', 'productivity']].mean())

In [ ]:
# Fit regression models
# Group 1 regression
X1_reg = sm.add_constant(df[df.skill_group == 1]['training_hours'])
model1 = sm.OLS(df[df.skill_group == 1]['productivity'], X1_reg).fit()

# Group 2 regression  
X2_reg = sm.add_constant(df[df.skill_group == 0]['training_hours'])
model2 = sm.OLS(df[df.skill_group == 0]['productivity'], X2_reg).fit()

# Combined regression
X_combined_reg = sm.add_constant(df['training_hours'])
model_combined = sm.OLS(df['productivity'], X_combined_reg).fit()

print(f"Group 1 (High-skill) slope: {model1.params[1]:.3f}")
print(f"Group 2 (Low-skill) slope: {model2.params[1]:.3f}")
print(f"Combined slope: {model_combined.params[1]:.3f}")
print(f"\nSimpsons paradox demonstrated: Individual groups show positive relationship ({model1.params[1]:.3f}, {model2.params[1]:.3f})")
print(f"but combined data shows negative relationship ({model_combined.params[1]:.3f})")

In [ ]:
# Create the Simpson's Paradox plot
plt.figure(figsize=(12, 8))

# Plot data points by group
high_skill = df[df.skill_group == 1]
low_skill = df[df.skill_group == 0]

plt.scatter(high_skill['training_hours'], high_skill['productivity'], 
           alpha=0.6, c='blue', label='High-skill workers', s=60)
plt.scatter(low_skill['training_hours'], low_skill['productivity'], 
           alpha=0.6, c='red', label='Low-skill workers', s=60)

# Create regression lines
x_range = np.linspace(df['training_hours'].min(), df['training_hours'].max(), 100)

# Group 1 regression line
y1_pred = model1.params[0] + model1.params[1] * x_range
plt.plot(x_range, y1_pred, 'blue', linewidth=2, linestyle='--', 
         label=f'High-skill regression (slope: {model1.params[1]:.2f})')

# Group 2 regression line
y2_pred = model2.params[0] + model2.params[1] * x_range
plt.plot(x_range, y2_pred, 'red', linewidth=2, linestyle='--',
         label=f'Low-skill regression (slope: {model2.params[1]:.2f})')

# Combined regression line
y_combined_pred = model_combined.params[0] + model_combined.params[1] * x_range
plt.plot(x_range, y_combined_pred, 'black', linewidth=3, 
         label=f'Combined regression (slope: {model_combined.params[1]:.2f})')

plt.xlabel('Training Hours')
plt.ylabel('Productivity Score')
plt.title("Simpson's Paradox: Training Hours vs Productivity\n" +
          "Positive relationship within groups, negative relationship overall")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{output_dir}/simpsons_paradox.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Save simulation results
df.to_csv(f'{output_dir}/simpsons_paradox_data.csv', index=False)

# Save regression results
results_summary = pd.DataFrame({
    'Model': ['High-skill Group', 'Low-skill Group', 'Combined'],
    'Slope': [model1.params[1], model2.params[1], model_combined.params[1]],
    'Intercept': [model1.params[0], model2.params[0], model_combined.params[0]],
    'R_squared': [model1.rsquared, model2.rsquared, model_combined.rsquared]
})

results_summary.to_csv(f'{output_dir}/simpsons_paradox_results.csv', index=False)
print("\nRegression Results Summary:")
print(results_summary)

## Summary

### Part 1a Summary
- **Confounder**: Ability confounds the education-income relationship
- **Collider**: Graduate school admission is a collider between talent and success
- **Mediator**: Labor costs mediate the minimum wage-unemployment relationship

### Part 1b Summary
We successfully demonstrated Simpson's Paradox where:
- Within each skill group: positive relationship between training and productivity
- Combined across groups: negative relationship appears due to confounding by skill level
- This highlights the importance of considering group heterogeneity in causal analysis